# exominer++ PCs

https://zenodo.org/records/15466293

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# df_exominer = pd.read_csv('../data/exominerplusplus_catalog_labeled_tces_s1-s67_tess-spoc-2min_complete_1-14-2025_1039.csv')
df_exominer = pd.read_csv('../data/exominerplusplus_catalog_unk_tces_s1-s67_tess-spoc-2min_complete_1-16-2025_1014.csv')
df_exominer.tail()

In [ ]:
idx = (df_exominer['Gaia RUWE']<2) & \
    (df_exominer['ExoMiner Score']>0.9) & \
    (df_exominer['Transit Depth [ppm]']>1000) & \
    (df_exominer['Transit Model SNR']>10) &\
    (df_exominer['Number of transits observed']>2) #&\
df_exominer[idx].shape

# get TIC catalog data containing Gaia ID for exominer TCEs

In [ ]:
from astroquery.mast import Catalogs

tic_data = Catalogs.query_criteria(catalog="TIC", ID=df_exominer.loc[idx, 'TIC ID'].unique())

In [ ]:
# Gaia DR3 source_id is a ~19-digit integer beyond float64's 2**53 exact-integer
# range. `Table.to_pandas()` silently loses precision for *masked* int64 columns
# (it round-trips through float64 internally), so build this column directly
# from the raw astropy column instead of trusting the pandas conversion.
tic_df = tic_data.to_pandas()
tic_df['GAIA'] = pd.array(
    [int(gid) if gid is not np.ma.masked else pd.NA for gid in tic_data['GAIA']],
    dtype='Int64',
)
tic_df.to_csv('../data/exominer_tic_data.csv', index=False)
tic_df.shape

In [ ]:
df_exominer_tic_data = pd.read_csv('../data/exominer_tic_data.csv', dtype={'GAIA': 'Int64'})
df_exominer_tic_data.head()

In [ ]:
# Extract unique Gaia IDs, dropping TIC entries with no Gaia cross-ID.
exominer_gaia_ids = df_exominer_tic_data['GAIA'].dropna().astype(int).unique().tolist()
len(exominer_gaia_ids)

# exominer PC but not TOI

* [Gaia DR3 1968083349388709120: also CTOI i.e. Bouma candidate](https://exofop.ipac.caltech.edu/tess/target.php?id=Gaia-DR3-1968083349388709120) <-- sometimes V-shaped; nearby@r=1.6"
* [Gaia DR3 2015547715865468288/ TIC 271759706](https://exofop.ipac.caltech.edu/tess/target.php?id=Gaia-DR3-2015547715865468288) <-- V-shaped based on ql
* [Gaia DR3 663140171661012480/ TIC 178247259](https://exofop.ipac.caltech.edu/tess/target.php?id=Gaia-DR3-663140171661012480) <-- Prot is an alias of Porb